# HealthSense: Data Mining & Predictive Modeling
### Module 3: Multi-Model Classification, Patient Clustering & Association Rule Mining

This notebook demonstrates 5 supervised classifiers (preventing data leakage), unsupervised K-Means clustering (elbow and silhouette score analysis), and Apriori association rule mining.


In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

import pandas as pd
import numpy as np

from src.classification import HealthClassifierTrainer
from src.clustering import compute_clustering_metrics, perform_kmeans_clustering
from src.association_rules import mine_association_rules
from src.explainability import get_global_feature_importance


## 1. Supervised Classification (5 Algorithms)
Train Logistic Regression, Decision Tree, Random Forest, KNN, and SVM on identical stratified train-test splits without data leakage.


In [ ]:
df = pd.read_csv('../data/processed/diabetes_cleaned.csv')
trainer = HealthClassifierTrainer(test_size=0.2, random_state=42)
results_df, artifacts = trainer.train_and_evaluate_all(df, target_column='Outcome')
print('\n=== CLASSIFIER BENCHMARK COMPARISON ===')
print(results_df.to_string(index=False))


## 2. Global Feature Importance
Identify the top predictive biomarkers using tree Gini importance or permutation importance.


In [ ]:
imp_df = get_global_feature_importance(
    artifacts['best_model'],
    artifacts['feature_names'],
    artifacts['X_test'],
    artifacts['y_test']
)
print('\nFeature Importances:')
print(imp_df.sort_values(by='Importance', ascending=False).to_string(index=False))


## 3. Unsupervised Clustering (K-Means)
Determine optimal clusters via Elbow Method & Silhouette Scores, and profile patient phenotypes.


In [ ]:
features_clust = ['Age', 'BMI', 'Glucose', 'BloodPressure']
clust_metrics = compute_clustering_metrics(df, features_clust, max_k=5)
print('Optimal k determined by Silhouette Score:', clust_metrics['optimal_k'])
print('Silhouette Scores for k=2..5:', clust_metrics['silhouettes'])

df_clustered, summary = perform_kmeans_clustering(df, features_clust, n_clusters=3)
print('\nCluster Profiles:')
print(summary['profiles'])


## 4. Association Rule Mining (Apriori)
Discretize health features into clinical categories and uncover frequent pattern associations with Disease Risk.


In [ ]:
freq_sets, rules = mine_association_rules(df, min_support=0.08, min_confidence=0.40)
print(f'Frequent Itemsets Discovered: {len(freq_sets)}')
print(f'Association Rules Extracted: {len(rules)}')
if not rules.empty:
    print(rules[['Antecedents', 'Consequents', 'support', 'confidence', 'lift']].head(10).to_string(index=False))
